In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
# ============================================================
# Fine-Tuning-PS-LLM
# Notebook: 05_dpo.ipynb
# Purpose: Fine-tune the SFT model using Direct Preference Optimization (DPO).
# ============================================================

In [3]:
# ============================================================
# Project Paths
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Colab Notebooks/Fine-Tuning-PS-LLM"
)

print(PROJECT_ROOT)
print(PROJECT_ROOT.exists())

/content/drive/MyDrive/Colab Notebooks/Fine-Tuning-PS-LLM
True


In [4]:
# ============================================================
# Mount Google Drive
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# ============================================================
# Add Project to Python Path
# ============================================================

import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(sys.path[-1])

/content/drive/MyDrive/Colab Notebooks/Fine-Tuning-PS-LLM


In [18]:
# ============================================================
# Configuration
# ============================================================
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"  # ← أعدها للـ 1.5B
#MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
#MODEL_NAME = "Qwen/Qwen2-0.5B-Instruct"  # ← بدل Qwen2.5-1.5B-Instruct
DPO_DATASET = (
    PROJECT_ROOT / "data" / "dpo" / "train.jsonl"
)

SFT_MODEL_PATH = (
    PROJECT_ROOT / "models" / "sft_qwen"
)

OUTPUT_DIR = (
    PROJECT_ROOT / "models" / "dpo_qwen"
)

NUM_EPOCHS = 2
LEARNING_RATE = 5e-6
BATCH_SIZE = 1
GRADIENT_ACCUMULATION = 8
MAX_LENGTH = 512  # ← غيّرناها من 2048
RANDOM_STATE = 42

In [7]:
# ============================================================
# Verify Project Files
# ============================================================

print("Project:", PROJECT_ROOT.exists())

print("Dataset:", DPO_DATASET.exists())

print("SFT Adapter:", SFT_MODEL_PATH.exists())

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Output:", OUTPUT_DIR)

Project: True
Dataset: True
SFT Adapter: True
Output: /content/drive/MyDrive/Colab Notebooks/Fine-Tuning-PS-LLM/models/P02_dpo


In [8]:
# ============================================================
# Install Dependencies
# ============================================================

!pip install -q \
trl \
peft \
accelerate \
datasets \
bitsandbytes>=0.46.1

In [9]:
# ============================================================
# Imports
# ============================================================

import os
import torch

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)

from peft import PeftModel

from trl import (
    DPOTrainer,
    DPOConfig,
)

In [10]:
import trl
import transformers
import peft
import accelerate

print("trl:", trl.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)

trl: 1.7.0
transformers: 5.12.0
peft: 0.19.1
accelerate: 1.14.0


In [12]:
# ============================================================
# Load SFT Model
# ============================================================

from transformers import BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, PeftModel, get_peft_model, LoraConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 1. حمّل النموذج الأساسي
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

# 2. حمّل SFT adapter وادمجه
sft_model = PeftModel.from_pretrained(base_model, SFT_MODEL_PATH)
model = sft_model.merge_and_unload()

# 3. جهّز للتدريب الكمّي
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=False,
)

# 4. أنشئ LoRA جديد صغير للـ DPO
lora_config = LoraConfig(
    r=4,
    lora_alpha=8,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
model.config.use_cache = False
model.train()

print("SFT model loaded successfully.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


trainable params: 544,768 || all params: 1,544,259,072 || trainable%: 0.0353
SFT model loaded successfully.


In [13]:
# ============================================================
# Load and Reformat Dataset
# ============================================================

from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files=str(DPO_DATASET),
    split="train",
)

MAX_PROMPT_TOKENS = 200
MAX_RESPONSE_TOKENS = 300  # prompt + response = 500 < 512

def format_dpo_sample(example):
    # قصّر الـ prompt
    prompt_ids = tokenizer(
        example["prompt"],
        max_length=MAX_PROMPT_TOKENS,
        truncation=True,
    )
    short_prompt = tokenizer.decode(
        prompt_ids["input_ids"],
        skip_special_tokens=True,
    )

    # قصّر الـ response
    chosen_ids = tokenizer(
        example["chosen"],
        max_length=MAX_RESPONSE_TOKENS,
        truncation=True,
    )
    short_chosen = tokenizer.decode(
        chosen_ids["input_ids"],
        skip_special_tokens=True,
    )

    rejected_ids = tokenizer(
        example["rejected"],
        max_length=MAX_RESPONSE_TOKENS,
        truncation=True,
    )
    short_rejected = tokenizer.decode(
        rejected_ids["input_ids"],
        skip_special_tokens=True,
    )

    return {
        "prompt": short_prompt,
        "chosen": short_prompt + "\n\n" + short_chosen,
        "rejected": short_prompt + "\n\n" + short_rejected,
    }

dataset = dataset.map(format_dpo_sample)

# تحقق
c_ids = tokenizer(dataset[0]["chosen"])["input_ids"]
r_ids = tokenizer(dataset[0]["rejected"])["input_ids"]
print("Chosen len:", len(c_ids))
print("Rejected len:", len(r_ids))

for i, (c, r) in enumerate(zip(c_ids, r_ids)):
    if c != r:
        print("First difference at token:", i)
        break

Map:   0%|          | 0/52 [00:00<?, ? examples/s]

Chosen len: 364
Rejected len: 371
First difference at token: 213


In [14]:
# ============================================================
# Configure DPO Training
# ============================================================

training_args = DPOConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    gradient_checkpointing=False,
    logging_steps=5,
    save_strategy="epoch",
    report_to="none",
    remove_unused_columns=False,
    beta=0.1,
    max_length= 512,
)

In [15]:
# ============================================================
# Create DPO Trainer
# ============================================================

trainer = DPOTrainer(

    model=model,

    ref_model=None,       # ← هذا يوفر ~50% من الذاكرة

    args=training_args,

    processing_class=tokenizer,

    train_dataset=dataset,
)

print("DPO Trainer created successfully.")

Adding EOS to train dataset:   0%|          | 0/52 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/52 [00:00<?, ? examples/s]

DPO Trainer created successfully.


In [16]:
# ============================================================
# Train DPO Model
# ============================================================

trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
5,0.690702
10,0.698629


TrainOutput(global_step=14, training_loss=0.6927377837044852, metrics={'train_runtime': 500.0051, 'train_samples_per_second': 0.208, 'train_steps_per_second': 0.028, 'total_flos': 816695745036288.0, 'train_loss': 0.6927377837044852, 'entropy': 0.8877534972769874, 'num_tokens': 103498.0, 'logits/chosen': 0.029352601264978028, 'logits/rejected': 0.01580447128797811, 'mean_token_accuracy': 0.8032574823924473, 'rewards/chosen': 0.0065058024733194286, 'rewards/rejected': 0.0001865931345881628, 'rewards/accuracies': 0.5714285714285714, 'rewards/margins': 0.0063192090352198905, 'logps/chosen': -359.8330764770508, 'logps/rejected': -359.37991224016463, 'epoch': 2.0})

In [17]:
# ============================================================
# Save DPO Model
# ============================================================

trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

print("Model saved to:", OUTPUT_DIR)

Model saved to: /content/drive/MyDrive/Colab Notebooks/Fine-Tuning-PS-LLM/models/P02_dpo
